# Thoughtful Learning writing-rating experiment
Select **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. No file upload is required: this notebook downloads the project code and collects the examples directly from Thoughtful Learning.

The source contained 208 examples at preparation time, with native Strong/Good/Okay/Poor labels. Related example sets remain together across train/validation/test splits. This trains a Qwen2.5-1.5B-Instruct QLoRA adapter for rating prediction, not a validated numerical grader or tutoring model. Commercial use relies on the project user's confirmation of authority to license these materials; original published license notices are retained. A different user must establish their own applicable permission before running. No model is deployed or published automatically. Free GPU availability is not guaranteed.

Software and collection checks passed locally; GPU training has not run in the development environment. Download results before your Colab session ends.


In [ ]:
import os, sys, pathlib, subprocess, tempfile, json
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Select a T4 GPU runtime before continuing.")
print("GPU:", torch.cuda.get_device_name(0))
# Each run gets its own folder, preserving previous runs and their results.
repo = pathlib.Path(tempfile.mkdtemp(prefix="thoughtful-", dir="/content")) / "essaygrader"
subprocess.run(["git", "clone", "--depth", "1", "--branch", "main",
    "https://github.com/sfstast/essaygrader.git", str(repo)], check=True)
os.chdir(repo)
print("Code revision:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-training.txt"], check=True)
prepared = "model_work/prepared/thoughtful-v1"
subprocess.run([sys.executable, "-m", "model_work.thoughtful_learning", "--authorized-use",
    "--out", prepared], check=True)
print("Working directory:", repo)


## Verify data and choose the model
The collector preserves source URLs, original license notices, source HTML and the project user's authorization attestation. It keeps related examples and exact duplicates in the same split. Test data stays out of training and model selection. Collection failures stop the run rather than silently omitting examples.


In [ ]:
from model_work.thoughtful_learning import preflight, messages, LABELS
from model_work.train_lora import tokenize_example
from huggingface_hub import HfApi
from transformers import AutoTokenizer
prepared = "model_work/prepared/thoughtful-v1"
summary, splits = preflight(prepared)
print(json.dumps(summary, indent=2))
model_id = "Qwen/Qwen2.5-1.5B-Instruct"
revision = HfApi().model_info(model_id).sha
tokenizer = AutoTokenizer.from_pretrained(model_id, revision=revision, trust_remote_code=False)
# Measure complete examples, never silently truncate writing or its label.
lengths = [len(tokenizer.apply_chat_template(messages(row), tokenize=True))
           for name in ("train", "validation") for row in splits[name]]
max_length = max(512, ((max(lengths) + 127) // 128) * 128)
if max_length > 8192:
    raise RuntimeError("Full samples exceed this bounded T4 recipe. Review context/memory settings; no samples were truncated.")
for name in ("train", "validation"):
    for row in splits[name]: tokenize_example(tokenizer, messages(row), max_length)
print("Full-example context length:", max_length)
pathlib.Path("model-revision.json").write_text(json.dumps({"model":model_id,"revision":revision}, indent=2))


## Train the adapter
Three epochs, 4-bit weights, batch size 1 and gradient accumulation. The trainer verifies the pinned model license and masks input tokens from the training loss. This may take a while on a free T4. An interrupted run is not a completed training result.

In [ ]:
from datetime import datetime, timezone
run_name = "thoughtful-" + datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S-%f")
output = pathlib.Path("model_work/adapters") / run_name
subprocess.run([sys.executable, "-m", "model_work.train_lora", "--thoughtful",
    "--prepared", prepared, "--train", "--model", model_id, "--revision", revision,
    "--out", str(output), "--qlora", "--epochs", "3", "--max-length", str(max_length)], check=True)
assert json.loads((output / "training-summary.json").read_text())["training_performed"]
print("Finished training:", output)


## Compare base versus adapter on the held-out examples
Both models receive identical writing without its title, source URL, or rating. Invalid outputs count as incorrect. Accuracy and macro F1 describe this small published-example set only; they do not establish classroom reliability or absence of pretraining overlap. Do not tune on these results and reuse the same test set as independent evidence.

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from contextlib import nullcontext
from collections import Counter
base = AutoModelForCausalLM.from_pretrained(model_id, revision=revision, trust_remote_code=False,
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16),
    device_map={"":torch.cuda.current_device()})
model = PeftModel.from_pretrained(base, str(output / "adapter"))
model.eval()
def predict(row, tuned):
    tokens = tokenizer.apply_chat_template(messages(row)[:-1], add_generation_prompt=True,
        return_tensors="pt").to(model.device)
    with torch.inference_mode(), (nullcontext() if tuned else model.disable_adapter()):
        result = model.generate(tokens, attention_mask=torch.ones_like(tokens), max_new_tokens=32,
            do_sample=False, pad_token_id=tokenizer.eos_token_id)
    text = tokenizer.decode(result[0,tokens.shape[1]:], skip_special_tokens=True)
    try:
        label = json.loads(text)["rating"]
        if label not in LABELS: label = None
    except (ValueError, TypeError, KeyError): label = None
    return label, text
reviews=[]
for row in splits["test"]:
    b, bt = predict(row, False); t, tt = predict(row, True)
    reviews.append({"id":row["id"], "expected":row["rating"], "base":b, "adapter":t,
        "base_output":bt, "adapter_output":tt, "source_url":row["source_url"]})
def metrics(key):
    correct = sum(r[key] == r["expected"] for r in reviews)
    f1=[]
    for label in LABELS:
        tp=sum(r[key]==label and r["expected"]==label for r in reviews)
        fp=sum(r[key]==label and r["expected"]!=label for r in reviews)
        fn=sum(r[key]!=label and r["expected"]==label for r in reviews)
        f1.append(2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0)
    return {"accuracy":correct/len(reviews), "macro_f1":sum(f1)/len(f1),
            "invalid_outputs":sum(r[key] is None for r in reviews), "examples":len(reviews)}
result={key:metrics(key) for key in ("base","adapter")}
majority=Counter(r["rating"] for r in splits["train"]).most_common(1)[0][0]
result["training_majority_baseline"]={"label":majority,
    "accuracy":sum(r["expected"]==majority for r in reviews)/len(reviews)}
(output / "test-results.json").write_text(json.dumps({"metrics":result,"predictions":reviews}, indent=2))
print(json.dumps(result, indent=2))


## Download results
Keep the adapter, exact model revision, original data manifest and attribution together. The website is not changed by this experiment. Review held-out errors before deciding whether to use the model.

In [ ]:
from google.colab import files
import shutil
bundle = pathlib.Path(tempfile.mkdtemp(prefix="thoughtful-results-", dir="/content"))
shutil.copytree(output, bundle / "training")
for name in ("manifest.json", "ATTRIBUTION.txt"):
    shutil.copy2(pathlib.Path(prepared) / name, bundle / name)
shutil.copy2("model-revision.json", bundle / "model-revision.json")
(bundle / "packages.txt").write_text(subprocess.check_output([sys.executable,"-m","pip","freeze"], text=True))
archive = shutil.make_archive(str(bundle), "zip", bundle)
files.download(archive)
